# 要解决的问题

- 如何使用FAISS存储向量？
- 如何使用FAISS进行向量相似度搜索？

# FAISS

官方仓库：[https://github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)

官网wiki：[https://github.com/facebookresearch/faiss/wiki](https://github.com/facebookresearch/faiss/wiki)

## 安装FAISS

In [1]:
!pip show faiss-cpu

Name: faiss-cpu
Version: 1.8.0
Summary: A library for efficient similarity search and clustering of dense vectors.
Home-page: 
Author: 
Author-email: Kota Yamaguchi <yamaguchi_kota@cyberagent.co.jp>
License: MIT License
Location: d:\software\anaconda3\envs\py310_llmappcourse2\lib\site-packages
Requires: numpy
Required-by: 


## 运行

以下代码来自于 [faiss/wiki/Getting-started](https://github.com/facebookresearch/faiss/wiki/Getting-started)

### Step 1: 生成数据

创建一些数据，用来作为数据库向量和查询向量

In [1]:
import numpy as np

# 设置向量的维度和数量
d = 64                          # 维度
nb = 100000                     # 数据库大小
nq = 10000                      # 查询数量

# 为了结果可重复，设置随机种子
np.random.seed(1234)

xb 表示数据库，其中包含我们要搜索的所有必须索引的向量。其大小为 nb 乘 d

In [2]:
# 创建数据库向量
xb = np.random.random((nb, d)).astype('float32')

In [3]:
xb.shape

(100000, 64)

In [4]:
xb

array([[0.19151945, 0.62210876, 0.43772775, ..., 0.62491673, 0.4780938 ,
        0.19567518],
       [0.38231745, 0.05387368, 0.4516484 , ..., 0.15139526, 0.33517465,
        0.65755177],
       [0.07334255, 0.0550064 , 0.3231948 , ..., 0.34441698, 0.6408804 ,
        0.12620533],
       ...,
       [0.81447345, 0.5902453 , 0.7988935 , ..., 0.339859  , 0.3019495 ,
        0.85385454],
       [0.6714615 , 0.9160688 , 0.9550788 , ..., 0.59536433, 0.03849181,
        0.10563799],
       [0.85663575, 0.59113413, 0.67890793, ..., 0.21897699, 0.06530159,
        0.21753833]], dtype=float32)

In [5]:
xb[:, 0]

array([0.19151945, 0.38231745, 0.07334255, ..., 0.81447345, 0.6714615 ,
       0.85663575], dtype=float32)

In [6]:
len(xb[:, 0])

100000

In [7]:
np.arange(nb)

array([    0,     1,     2, ..., 99997, 99998, 99999])

In [10]:
np.arange(nb) / 1000.

array([0.0000e+00, 1.0000e-03, 2.0000e-03, ..., 9.9997e+01, 9.9998e+01,
       9.9999e+01])

In [11]:
# 0; 0.001; 0.002; 0.003; ... 99.98; 99.999

In [10]:
# 为 xb 的第一列加上一个与索引相关的小递增量，引入一个轻微的线性趋势。
# 这一步是可选的，目的只是为了增加数据的变化性。
# 执行这一步后，xb 的第一列数值随索引递增，其他列的显示格式可能会因数值大小的自动调整而从十进制转为科学记数法。
xb[:, 0] += np.arange(nb) / 1000.

In [8]:
# 第一行第一列加0
# 第二行第一列加0.001
# 第三行第一列加0.002
# 第四行第一列加0.003
# 倒数第二行第一列加99.98
# 倒数第一行第一列加99.99

In [11]:
xb

array([[1.91519454e-01, 6.22108757e-01, 4.37727749e-01, ...,
        6.24916732e-01, 4.78093803e-01, 1.95675179e-01],
       [3.83317441e-01, 5.38736843e-02, 4.51648414e-01, ...,
        1.51395261e-01, 3.35174650e-01, 6.57551765e-01],
       [7.53425434e-02, 5.50063960e-02, 3.23194802e-01, ...,
        3.44416976e-01, 6.40880406e-01, 1.26205325e-01],
       ...,
       [1.00811470e+02, 5.90245306e-01, 7.98893511e-01, ...,
        3.39859009e-01, 3.01949501e-01, 8.53854537e-01],
       [1.00669464e+02, 9.16068792e-01, 9.55078781e-01, ...,
        5.95364332e-01, 3.84918079e-02, 1.05637990e-01],
       [1.00855637e+02, 5.91134131e-01, 6.78907931e-01, ...,
        2.18976989e-01, 6.53015897e-02, 2.17538327e-01]], dtype=float32)

xq 表示查询向量，我们需要为其找到最近的邻居。其大小为 nq 乘 d。如果只有一个查询向量，则 nq=1。

In [12]:
# 创建查询向量
xq = np.random.random((nq, d)).astype('float32')

In [13]:
xq

array([[0.81432974, 0.7409969 , 0.8915324 , ..., 0.72459674, 0.893881  ,
        0.6574571 ],
       [0.58347744, 0.797842  , 0.74140453, ..., 0.6768835 , 0.05907924,
        0.6396156 ],
       [0.74840766, 0.02659794, 0.5495097 , ..., 0.69562465, 0.16268532,
        0.76653737],
       ...,
       [0.97072977, 0.05037309, 0.7342035 , ..., 0.89510185, 0.6490696 ,
        0.86151606],
       [0.8331929 , 0.70606154, 0.1922274 , ..., 0.8026039 , 0.6854174 ,
        0.60209423],
       [0.07948319, 0.39106598, 0.01359335, ..., 0.63193923, 0.12561724,
        0.78384215]], dtype=float32)

In [14]:
xq.shape

(10000, 64)

In [15]:
# 为查询向量 xq 的第一列加上一个与索引相关的小递增量，引入轻微的线性趋势。
# 类似于 xb，这一步是可选的，用于增加查询数据的变化性。
# 它有助于模拟现实世界数据中可能存在的渐变特征或趋势。
xq[:, 0] += np.arange(nq) / 1000.

In [16]:
xq

array([[ 0.81432974,  0.7409969 ,  0.8915324 , ...,  0.72459674,
         0.893881  ,  0.6574571 ],
       [ 0.5844774 ,  0.797842  ,  0.74140453, ...,  0.6768835 ,
         0.05907924,  0.6396156 ],
       [ 0.75040764,  0.02659794,  0.5495097 , ...,  0.69562465,
         0.16268532,  0.76653737],
       ...,
       [10.96773   ,  0.05037309,  0.7342035 , ...,  0.89510185,
         0.6490696 ,  0.86151606],
       [10.831193  ,  0.70606154,  0.1922274 , ...,  0.8026039 ,
         0.6854174 ,  0.60209423],
       [10.078484  ,  0.39106598,  0.01359335, ...,  0.63193923,
         0.12561724,  0.78384215]], dtype=float32)

In [19]:
# 数组中的数值在一定范围内，因此打印时未采用科学记数法。
# Python 在打印浮点数时，会根据数值的大小自动选择显示格式。


### Step 2: 构建索引并向其添加向量

有了向量数据后，接着创建FAISS索引并将数据库向量添加到索引中

- 索引对象的构建：
  - FAISS 基于索引对象（Index object）构建，该对象封装了数据库向量的集合，并可以对这些向量进行可选的预处理，以提高搜索效率。
  - 索引对象支持多种类型，本例中使用的是 IndexFlatL2，这是最简单的索引类型，只进行暴力搜索 L2 距离。

- 索引的基本要求：
  - 构建索引时，需要指定操作向量的维度（在本例中为 d）。
  - 大多数索引类型需要一个训练阶段来分析向量的分布，但对于 IndexFlatL2，这一步骤可以省略。

In [17]:
d

64

In [18]:
import faiss

# 创建一个使用L2距离的Flat索引
index = faiss.IndexFlatL2(d)   # 指定向量维度

# 查看索引是否已经训练（对于Flat索引，这一步不是必需的，应该显示True）
print(index.is_trained)

True


- 索引操作：
    - 构建并训练（如果需要）索引后，可以在索引上执行两种操作：添加（add）和搜索（search）。
    - 添加向量到索引，通过对向量集 xb 调用 add 方法实现。
    - 索引还维护两个状态变量：is_trained（布尔值，表示是否需要训练）和 ntotal（索引中的向量数量）。

In [19]:
xb.shape

(100000, 64)

In [20]:
# 将数据库向量添加到索引
index.add(xb)
print(index.ntotal)

100000


In [21]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001C80033DE00> >

### Step 3: 搜索

使用索引来找到每个查询向量的最近邻

- 基本搜索操作：
    - 基本的搜索操作是 k-最近邻（k-Nearest Neighbor, k-NN）搜索，即对于每个查询向量，找到数据库中与之最近的 k 个邻居向量。

- 搜索结果的存储：
    - 搜索的结果可以方便地存储在一个整数矩阵（I）中，该矩阵的大小为 nq 乘 k，其中第 i 行包含查询向量 i 的邻居的 ID，按距离递增排序。
    - 除了整数矩阵，搜索操作还返回一个 nq 乘 k 的浮点矩阵（D），包含相应的平方距离。

- 健全性检查：
    - 为了验证搜索的准确性，可以先对一些数据库中的向量进行搜索，以确保最近的邻居确实是向量本身。

In [23]:
xb[:5].shape

(5, 64)

In [25]:
xb[1]

array([0.38331744, 0.05387368, 0.4516484 , 0.98200476, 0.1239427 ,
       0.1193809 , 0.73852307, 0.58730364, 0.47163254, 0.10712682,
       0.22921857, 0.89996517, 0.41675353, 0.53585166, 0.00620852,
       0.30064172, 0.43689317, 0.612149  , 0.91819805, 0.62573665,
       0.7059976 , 0.14983371, 0.7460634 , 0.831007  , 0.63372576,
       0.43830988, 0.15257278, 0.5684096 , 0.5282243 , 0.9514288 ,
       0.48035917, 0.50255954, 0.53687817, 0.81920207, 0.05711564,
       0.66942173, 0.7671166 , 0.70811534, 0.7968672 , 0.55776083,
       0.9658365 , 0.1471569 , 0.029647  , 0.59389347, 0.1140657 ,
       0.95080984, 0.3257074 , 0.19361869, 0.45781165, 0.9204026 ,
       0.87906915, 0.25261575, 0.34800878, 0.18258873, 0.90179604,
       0.7065282 , 0.72665846, 0.90008783, 0.7791638 , 0.5991548 ,
       0.29112524, 0.15139526, 0.33517465, 0.65755177], dtype=float32)

In [22]:
k = 4  # 定义每个查询返回的最近邻的数量

# 对于数据库中的前5个向量进行健全性检查，确认最近邻是它们自己
D, I = index.search(xb[:5], k)
print(I)
print(D)

[[  0 393 363  78]
 [  1 555 277 364]
 [  2 304 101  13]
 [  3 173  18 182]
 [  4 288 370 531]]
[[0.        7.1751733 7.2076297 7.2511625]
 [0.        6.3235645 6.684581  6.799946 ]
 [0.        5.7964087 6.391736  7.2815123]
 [0.        7.2779055 7.5279875 7.662846 ]
 [0.        6.7638035 7.2951202 7.368815 ]]


In [26]:
D.shape

(5, 4)

In [27]:
I.shape

(5, 4)

In [28]:
xq.shape

(10000, 64)

In [29]:
# 实际对查询向量进行搜索
D, I = index.search(xq, k)
print(I[:5])
print(I[-5:])

[[ 381  207  210  477]
 [ 526  911  142   72]
 [ 838  527 1290  425]
 [ 196  184  164  359]
 [ 526  377  120  425]]
[[ 9900 10500  9309  9831]
 [11055 10895 10812 11321]
 [11353 11103 10164  9787]
 [10571 10664 10632  9638]
 [ 9628  9554 10036  9582]]


In [30]:
print(I[:5])
print(D[:5])

[[ 381  207  210  477]
 [ 526  911  142   72]
 [ 838  527 1290  425]
 [ 196  184  164  359]
 [ 526  377  120  425]]
[[6.8155136 6.8894653 7.3956795 7.4290257]
 [6.6041107 6.679699  6.7209625 6.828678 ]
 [6.4703865 6.8578606 7.0043755 7.036564 ]
 [5.5736923 6.407551  7.1395226 7.3555984]
 [5.409401  6.232216  6.4173393 6.57436  ]]


In [26]:
D.shape

(10000, 4)

In [27]:
I.shape

(10000, 4)